In [ ]:
import os
_cuda_bin = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0\bin'
if os.path.isdir(_cuda_bin):
    os.add_dll_directory(_cuda_bin)
    if os.path.isdir(_cuda_bin + r'\x64'):
        os.add_dll_directory(_cuda_bin + r'\x64')

import numpy as np
import json
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import mean_squared_error as mse_metric
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
MANIFEST_PATH = 'ScAlN_dataset_3class/split_manifest.json'
FRAMES_DIR = 'dataset_prediction/average_frames'
SEED = 2025
MAX_EPOCHS = 100
BATCH_SIZE = 4
LR = 0.0005
WARMUP_EPOCHS = 5
INPUT_LEN = 15
OUTPUT_LEN = 5
IMG_SIZE = (128, 128)
PATIENCE = 10
WEIGHT_DECAY = 1e-4
MEMORY_MAX_NORM = 10.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

In [ ]:
class RHEEDBlockDataset(Dataset):
    def __init__(self, frames_dir, manifest_path, split='train',
                 input_len=INPUT_LEN, output_len=OUTPUT_LEN, img_size=IMG_SIZE,
                 augment=False):
        self.frames_dir = Path(frames_dir)
        self.input_len = input_len
        self.output_len = output_len
        self.total_len = input_len + output_len
        self.img_size = img_size
        self.augment = augment

        all_frames = sorted(self.frames_dir.glob('*.jpg'))
        if not all_frames:
            all_frames = sorted(self.frames_dir.glob('*.png'))
        self.all_frames = all_frames

        with open(manifest_path, 'r', encoding='utf-8') as f:
            manifest = json.load(f)

        split_seconds = sorted(manifest['splits'][split]['seconds'])

        contiguous_runs = []
        if split_seconds:
            run_start = split_seconds[0]
            prev = split_seconds[0]
            for sec in split_seconds[1:]:
                if sec == prev + 1:
                    prev = sec
                else:
                    contiguous_runs.append((run_start, prev))
                    run_start = sec
                    prev = sec
            contiguous_runs.append((run_start, prev))

        self.windows = []
        for rs, re in contiguous_runs:
            run_len = re - rs + 1
            if run_len >= self.total_len:
                for start in range(rs, re - self.total_len + 2):
                    self.windows.append(start)

        print(f"[{split}] {len(contiguous_runs)} runs, {len(self.windows)} windows, augment={augment}")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        start_sec = self.windows[idx]
        assert start_sec + self.total_len <= len(self.all_frames), (
            f'window start {start_sec} + total_len {self.total_len} exceeds '
            f'{len(self.all_frames)} available frames'
        )

        sequence = []
        for i in range(self.total_len):
            img_path = self.all_frames[start_sec + i]
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, self.img_size)
                img = img.astype(np.float32) / 255.0
            else:
                img = np.zeros(self.img_size, dtype=np.float32)
            sequence.append(img)

        sequence = np.array(sequence)

        if self.augment:

            brightness_factor = np.random.uniform(0.9, 1.1)
            sequence = np.clip(sequence * brightness_factor, 0, 1)
            noise = np.random.normal(0, 0.01, sequence.shape).astype(np.float32)
            sequence = np.clip(sequence + noise, 0, 1)
            if np.random.rand() < 0.5:
                crop_size = 112
                max_y = self.img_size[1] - crop_size
                max_x = self.img_size[0] - crop_size
                y0 = np.random.randint(0, max_y + 1)
                x0 = np.random.randint(0, max_x + 1)
                cropped = sequence[:, y0:y0+crop_size, x0:x0+crop_size]
                resized = np.zeros_like(sequence)
                for t in range(len(sequence)):
                    resized[t] = cv2.resize(cropped[t], self.img_size)
                sequence = resized

        sequence = sequence[:, np.newaxis, :, :]
        x = torch.FloatTensor(sequence[:self.input_len])
        y = torch.FloatTensor(sequence[self.input_len:])
        return x, y

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size, padding=kernel_size // 2)

    def forward(self, x, hidden_state):
        h, c = hidden_state
        gates = self.conv(torch.cat([x, h], dim=1))
        i, f, o, g = torch.split(gates, self.hidden_dim, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


class SelfAttentionMemory(nn.Module):
    def __init__(self, channels, reduction=4, spatial_reduction=8, memory_max_norm=MEMORY_MAX_NORM):
        super().__init__()
        self.channels = channels
        self.d_k = channels // reduction
        self.spatial_reduction = spatial_reduction
        self.memory_max_norm = memory_max_norm

        self.W_hq = nn.Conv2d(channels, self.d_k, 1)
        self.W_hk = nn.Conv2d(channels, self.d_k, 1)
        self.W_hv = nn.Conv2d(channels, channels, 1)
        self.W_mk = nn.Conv2d(channels, self.d_k, 1)
        self.W_mv = nn.Conv2d(channels, channels, 1)
        self.W_z = nn.Conv2d(channels * 2, channels, 1)
        self.attn_dropout = nn.Dropout2d(0.1)
        self.W_mg = nn.Conv2d(channels, channels, 1)
        self.W_mo = nn.Conv2d(channels, channels, 1)
        self.W_mi = nn.Conv2d(channels, channels, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, h_t, m_prev):
        B, C, H, W = h_t.shape
        sr = self.spatial_reduction

        h_small = F.adaptive_avg_pool2d(h_t, (H // sr, W // sr))
        m_small = F.adaptive_avg_pool2d(m_prev, (H // sr, W // sr))
        _, _, hs, ws = h_small.shape
        sp = hs * ws
        scale = np.sqrt(self.d_k)

        Q_h = self.W_hq(h_small).view(B, self.d_k, sp).permute(0, 2, 1)
        K_h = self.W_hk(h_small).view(B, self.d_k, sp)
        K_m = self.W_mk(m_small).view(B, self.d_k, sp)
        V_h = self.W_hv(h_small).view(B, C, sp)
        V_m = self.W_mv(m_small).view(B, C, sp)

        A_h = F.softmax(torch.bmm(Q_h, K_h) / scale, dim=-1)
        A_m = F.softmax(torch.bmm(Q_h, K_m) / scale, dim=-1)

        Z_h = torch.bmm(V_h, A_h.permute(0, 2, 1)).view(B, C, hs, ws)
        Z_m = torch.bmm(V_m, A_m.permute(0, 2, 1)).view(B, C, hs, ws)

        Z_h = F.interpolate(Z_h, size=(H, W), mode='bilinear', align_corners=False)
        Z_m = F.interpolate(Z_m, size=(H, W), mode='bilinear', align_corners=False)

        Z = self.W_z(torch.cat([Z_h, Z_m], dim=1))
        Z = self.attn_dropout(Z)

        output_gate = torch.sigmoid(self.W_mo(Z))
        h_t_hat = output_gate * torch.tanh(self.W_mg(Z))

        input_gate = torch.sigmoid(self.W_mi(Z))
        m_t = (1 - input_gate) * m_prev + input_gate * h_t_hat

        m_t_norm = torch.norm(m_t, dim=1, keepdim=True).clamp(min=1e-8)
        m_t = torch.where(m_t_norm > self.memory_max_norm,
                          m_t * self.memory_max_norm / m_t_norm, m_t)

        return h_t_hat, m_t


class MSAMConvLSTM(nn.Module):
    def __init__(self, input_channels=1, hidden_dims=[64, 128, 128],
                 kernel_size=3, num_layers=3, attention_reduction=4,
                 dropout_rates=[0.1, 0.1, 0.2]):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dims = hidden_dims

        self.encoder_cells = nn.ModuleList()
        self.encoder_sam = nn.ModuleList()
        self.encoder_bns = nn.ModuleList()
        self.encoder_drops = nn.ModuleList()
        self.decoder_cells = nn.ModuleList()
        self.decoder_sam = nn.ModuleList()
        self.decoder_bns = nn.ModuleList()
        self.decoder_drops = nn.ModuleList()

        for i in range(num_layers):
            inp = input_channels if i == 0 else hidden_dims[i-1]
            self.encoder_cells.append(ConvLSTMCell(inp, hidden_dims[i], kernel_size))
            self.encoder_sam.append(SelfAttentionMemory(hidden_dims[i], attention_reduction))
            self.encoder_bns.append(nn.BatchNorm2d(hidden_dims[i]))
            self.encoder_drops.append(nn.Dropout2d(dropout_rates[i]))
            self.decoder_cells.append(ConvLSTMCell(inp, hidden_dims[i], kernel_size))
            self.decoder_sam.append(SelfAttentionMemory(hidden_dims[i], attention_reduction))
            self.decoder_bns.append(nn.BatchNorm2d(hidden_dims[i]))
            self.decoder_drops.append(nn.Dropout2d(dropout_rates[i]))

        self.output_conv = nn.Conv2d(hidden_dims[-1], input_channels, 1)

    def forward(self, x, future_seq=5):
        B, T, C, H, W = x.shape

        states = [(torch.zeros(B, d, H, W, device=x.device),
                    torch.zeros(B, d, H, W, device=x.device))
                   for d in self.hidden_dims]
        memories = [torch.zeros(B, d, H, W, device=x.device) for d in self.hidden_dims]

        for t in range(T):
            inp = x[:, t]
            for i in range(self.num_layers):
                h, c = self.encoder_cells[i](inp if i == 0 else h_out, states[i])
                h_out, m_new = self.encoder_sam[i](h, memories[i])
                h_out = self.encoder_drops[i](self.encoder_bns[i](h_out))
                states[i] = (h_out, c)
                memories[i] = m_new

        dec_states = [(h.clone(), c.clone()) for h, c in states]
        dec_memories = [m.clone() for m in memories]
        dec_input = x[:, -1]
        outputs = []

        for t in range(future_seq):
            for i in range(self.num_layers):
                h, c = self.decoder_cells[i](dec_input if i == 0 else h_out, dec_states[i])
                h_out, m_new = self.decoder_sam[i](h, dec_memories[i])
                h_out = self.decoder_drops[i](self.decoder_bns[i](h_out))
                dec_states[i] = (h_out, c)
                dec_memories[i] = m_new
            out = torch.sigmoid(self.output_conv(h_out))
            outputs.append(out)
            dec_input = out
        return torch.stack(outputs, dim=1)

In [ ]:
def calculate_metrics(predictions, targets):
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()
    if torch.is_tensor(targets):
        targets = targets.detach().cpu().numpy()
    ssim_scores, mse_scores, mae_scores = [], [], []
    for b in range(predictions.shape[0]):
        for t in range(predictions.shape[1]):
            pred = predictions[b, t, 0]
            tgt = targets[b, t, 0]
            ssim_scores.append(ssim_metric(tgt, pred, data_range=1.0))
            mse_scores.append(mse_metric(tgt, pred))
            mae_scores.append(np.mean(np.abs(tgt - pred)))
    return {
        'SSIM': float(np.mean(ssim_scores)),
        'MSE': float(np.mean(mse_scores) * 1000),
        'MAE': float(np.mean(mae_scores) * 1000),
    }

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch_x, batch_y in tqdm(loader, desc='Training', leave=False):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred = model(batch_x, future_seq=batch_y.shape[1])
        mse_loss = F.mse_loss(pred, batch_y)
        rollout_mse = 1 - torch.mean(torch.stack([
            1 - F.mse_loss(pred[:, i], batch_y[:, i]) for i in range(pred.shape[1])
        ]))
        loss = mse_loss + 0.1 * rollout_mse
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    all_pred, all_tgt = [], []
    total_loss = 0
    with torch.no_grad():
        for bx, by in tqdm(loader, desc='Evaluating', leave=False):
            bx, by = bx.to(device), by.to(device)
            pred = model(bx, future_seq=by.shape[1])
            mse_loss = F.mse_loss(pred, by)
            rollout_mse = 1 - torch.mean(torch.stack([
                1 - F.mse_loss(pred[:, i], by[:, i]) for i in range(pred.shape[1])
            ]))
            loss = mse_loss + 0.1 * rollout_mse
            total_loss += loss.item()
            all_pred.append(pred)
            all_tgt.append(by)
    all_pred = torch.cat(all_pred, 0)
    all_tgt = torch.cat(all_tgt, 0)
    metrics = calculate_metrics(all_pred, all_tgt)
    metrics['Loss'] = total_loss / len(loader)
    return metrics


def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']


def train_model(model, model_name, result_dir, train_loader, val_loader, test_loader):
    os.makedirs(result_dir, exist_ok=True)
    best_path = os.path.join(result_dir, 'best_model.pth')

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"Training: {model_name}")
    print(f"Parameters: {total_params/1e6:.3f}M")
    print(f"Result dir: {result_dir}")
    print(f"{'='*70}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)

    best_val_ssim = 0
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_ssim': [], 'val_mse': [], 'val_mae': [], 'lr': []}

    train_start = time.time()

    for epoch in range(MAX_EPOCHS):
        if epoch < WARMUP_EPOCHS:
            warmup_lr = 1e-5 + (LR - 1e-5) * epoch / WARMUP_EPOCHS
            for pg in optimizer.param_groups:
                pg['lr'] = warmup_lr

        train_loss = train_epoch(model, train_loader, optimizer, device)
        history['train_loss'].append(float(train_loss))
        history['lr'].append(float(get_lr(optimizer)))

        val_m = evaluate(model, val_loader, device)
        history['val_loss'].append(float(val_m['Loss']))
        history['val_ssim'].append(float(val_m['SSIM']))
        history['val_mse'].append(float(val_m['MSE']))
        history['val_mae'].append(float(val_m['MAE']))

        if epoch % 5 == 0 or epoch == MAX_EPOCHS - 1:
            print(f"  Epoch [{epoch+1}/{MAX_EPOCHS}] loss={train_loss:.4f} | "
                  f"val_SSIM={val_m['SSIM']:.4f} MSE={val_m['MSE']:.1f} "
                  f"lr={get_lr(optimizer):.2e}")

        if val_m['SSIM'] > best_val_ssim:
            best_val_ssim = val_m['SSIM']
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_path)
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (patience={PATIENCE})")
            break

        if epoch >= WARMUP_EPOCHS:
            cosine_scheduler.step()

        torch.cuda.empty_cache()

    training_time = time.time() - train_start
    epochs_trained = epoch + 1
    print(f"  Training time: {training_time/60:.1f} min ({epochs_trained} epochs)")

    model.load_state_dict(torch.load(best_path, weights_only=True))
    test_m = evaluate(model, test_loader, device)
    print(f"  Test SSIM={test_m['SSIM']:.4f} MSE={test_m['MSE']:.1f} MAE={test_m['MAE']:.1f}")

    report = {
        'model_name': model_name,
        'test_SSIM': test_m['SSIM'],
        'test_MSE': test_m['MSE'],
        'test_MAE': test_m['MAE'],
        'best_val_SSIM': best_val_ssim,
        'total_params': total_params,
        'training_time_seconds': float(training_time),
        'training_time_minutes': float(training_time / 60),
        'epochs_trained': epochs_trained,
        'early_stopped': patience_counter >= PATIENCE,
    }
    with open(os.path.join(result_dir, 'test_report.json'), 'w') as f:
        json.dump(report, f, indent=2)
    with open(os.path.join(result_dir, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{model_name} - Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['val_ssim'])
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSIM')
    axes[1].set_title(f'{model_name} - Val SSIM'); axes[1].grid(True, alpha=0.3)
    axes[2].plot(history['val_mse'])
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE')
    axes[2].set_title(f'{model_name} - Val MSE'); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, 'training_curves.png'), dpi=300)
    plt.show()

    model.eval()
    with torch.no_grad():
        sx, sy = next(iter(test_loader))
        sx, sy = sx.to(device), sy.to(device)
        sp = model(sx, future_seq=sy.shape[1])
        fig, axes = plt.subplots(3, 5, figsize=(15, 9))
        for i in range(5):
            axes[0, i].imshow(sx[0, -5+i, 0].cpu(), cmap='gray')
            axes[0, i].set_title(f'Input t-{5-i}'); axes[0, i].axis('off')
            axes[1, i].imshow(sy[0, i, 0].cpu(), cmap='gray')
            axes[1, i].set_title(f'Truth t+{i+1}'); axes[1, i].axis('off')
            axes[2, i].imshow(sp[0, i, 0].cpu(), cmap='gray')
            axes[2, i].set_title(f'Pred t+{i+1}'); axes[2, i].axis('off')
        plt.suptitle(f'{model_name} Predictions (Test Set)')
        plt.tight_layout()
        plt.savefig(os.path.join(result_dir, 'predictions_sample.png'), dpi=150)
        plt.show()

    return report

In [ ]:
def main():
    print("Loading datasets...")
    train_ds = RHEEDBlockDataset(FRAMES_DIR, MANIFEST_PATH, 'train', augment=True)
    val_ds = RHEEDBlockDataset(FRAMES_DIR, MANIFEST_PATH, 'val', augment=False)
    test_ds = RHEEDBlockDataset(FRAMES_DIR, MANIFEST_PATH, 'test', augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    set_seed()
    model = MSAMConvLSTM().to(device)
    report = train_model(model, 'MSAM-ConvLSTM', 'results/msam_convlstm_v3',
                         train_loader, val_loader, test_loader)
    del model; torch.cuda.empty_cache()

    print("\nFinal Report:")
    for k, v in report.items():
        print(f"  {k}: {v}")
    return report


In [ ]:
if __name__ == "__main__":
    report = main()
